# 03 · OkTex Measurement Analytics (Delta)

Analytical queries over `stable_classic_wg38i9_catalog.oneok_okt` powering the dashboard and app — daily system balance, throughput by segment, and meters with the largest scheduled-vs-actual variance. Outputs are live query results.

In [ ]:
from dbx_sql import run_sql, fmt_table, get_token, CATALOG, SCHEMA
FQ = f'{CATALOG}.{SCHEMA}'
tok = get_token()
print('querying', FQ)

querying stable_classic_wg38i9_catalog.oneok_okt


### 1. Daily system balance — receipts vs deliveries (last 10 days)

In [ ]:
rows, cols = run_sql(f'''
  SELECT m.gas_day,
         ROUND(SUM(CASE WHEN d.meter_type='RECEIPT' THEN m.actual_dth END)/1000,1) AS receipts_mdth,
         ROUND(SUM(CASE WHEN d.meter_type IN ('DELIVERY','BIDIRECTIONAL') THEN m.actual_dth END)/1000,1) AS deliv_mdth,
         ROUND((SUM(CASE WHEN d.meter_type='RECEIPT' THEN m.actual_dth END)
               -SUM(CASE WHEN d.meter_type IN ('DELIVERY','BIDIRECTIONAL') THEN m.actual_dth END))/1000,1) AS imbalance_mdth
  FROM {FQ}.fact_daily_measurements m JOIN {FQ}.dim_meters d USING (meter_id)
  WHERE m.gas_day >= (SELECT MAX(gas_day) FROM {FQ}.fact_daily_measurements) - INTERVAL 9 DAYS
  GROUP BY m.gas_day ORDER BY m.gas_day''', tok)
print(fmt_table(rows, cols))

gas_day    | receipts_mdth | deliv_mdth | imbalance_mdth
-----------+---------------+------------+---------------
2026-08-19 | 542.7         | 534.0      | 8.7           
2026-08-20 | 561.4         | 564.1      | -2.6          
2026-08-21 | 553.6         | 561.6      | -8.0          
2026-08-22 | 507.2         | 504.9      | 2.3           
2026-08-23 | 505.2         | 512.8      | -7.6          
2026-08-24 | 540.5         | 544.7      | -4.2          
2026-08-25 | 562.3         | 560.6      | 1.7           
2026-08-26 | 558.4         | 557.4      | 1.0           
2026-08-27 | 578.7         | 564.6      | 14.1          
2026-08-28 | 536.3         | 536.9      | -0.6          


### 2. Throughput by pipeline segment (today)

In [ ]:
rows, cols = run_sql(f'''
  SELECT d.segment, COUNT(*) AS meters, ROUND(SUM(m.actual_dth)/1000,1) AS total_mdth
  FROM {FQ}.fact_daily_measurements m JOIN {FQ}.dim_meters d USING (meter_id)
  WHERE m.gas_day = (SELECT MAX(gas_day) FROM {FQ}.fact_daily_measurements)
  GROUP BY d.segment ORDER BY total_mdth DESC''', tok)
print(fmt_table(rows, cols))

segment              | meters | total_mdth
---------------------+--------+-----------
West Texas — El Paso | 17     | 388.2     
Red River Border     | 11     | 365.5     
Western Oklahoma     | 9      | 162.3     
Central Oklahoma     | 4      | 111.8     
Caprock Spur         | 2      | 45.4      


### 3. Meters with the largest measurement variance (today)

In [ ]:
rows, cols = run_sql(f'''
  SELECT d.meter_id, d.meter_name, d.meter_type, m.scheduled_dth, m.actual_dth, m.variance_pct
  FROM {FQ}.fact_daily_measurements m JOIN {FQ}.dim_meters d USING (meter_id)
  WHERE m.gas_day = (SELECT MAX(gas_day) FROM {FQ}.fact_daily_measurements)
  ORDER BY ABS(m.variance_pct) DESC LIMIT 10''', tok)
print(fmt_table(rows, cols))

meter_id   | meter_name         | meter_type    | scheduled_dth | actual_dth | variance_pct
-----------+--------------------+---------------+---------------+------------+-------------
OKT-OK9-01 | WesTex Hemphill IC | BIDIRECTIONAL | 18642         | 19731      | 5.84        
OKT-OK9-09 | PEPL Aledo         | DELIVERY      | 4996          | 4711       | -5.7        
OKT-OK3-03 | OK-3 CB (192137)   | DELIVERY      | 13543         | 12789      | -5.57       
OKT-O12-04 | Southern Star      | DELIVERY      | 15991         | 16865      | 5.46        
OKT-O11-02 | OGT Caprock        | BIDIRECTIONAL | 27297         | 25809      | -5.45       
OKT-DN5-04 | Frank Lehman       | BIDIRECTIONAL | 34735         | 32857      | -5.4        
OKT-OK3-01 | Jackson T. Wichita | RECEIPT       | 59053         | 55875      | -5.38       
OKT-DN5-05 | PNM-Anthony        | BIDIRECTIONAL | 14916         | 15681      | 5.13        
OKT-DN4-04 | El Paso Canutillo  | DELIVERY      | 16225         | 15472      | -